# Real vs AI Art Detection - Data Exploration
Task 2: Inspect the data and perform data understanding and preparation. If needed, perform appropriate down-sampling.

In [ ]:
!pip install kagglehub -q Pillow matplotlib

In [ ]:
import kagglehub
import os
from PIL import Image
import matplotlib.pyplot as plt
import shutil
import random

## 1. Download Dataset

In [ ]:
# Download latest version
dataset_path = kagglehub.dataset_download("ravidussilva/real-ai-art")
print("Path to dataset files:", dataset_path)

## 2. Inspect Dataset Structure

In [ ]:
def inspect_dataset(path):
    total_images = 0
    categories = {}
    for root, dirs, files in os.walk(path):
        img_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if img_files:
            # We assume a structure like train/real, train/fake, test/real, test/fake
            category = os.path.basename(root)
            parent = os.path.basename(os.path.dirname(root))
            name = f"{parent}/{category}"
            count = len(img_files)
            categories[name] = count
            total_images += count
            print(f"Found {count} images in {name}")
    print(f"\nTotal images: {total_images}")
    return categories

categories = inspect_dataset(dataset_path)

## 3. Visualize Sample Images
Displaying random sample images along with their resolution.

In [ ]:
def show_samples(path, num_samples=3):
    for root, dirs, files in os.walk(path):
        img_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if img_files:
            category = os.path.basename(root)
            parent = os.path.basename(os.path.dirname(root))
            print(f"\n--- Samples from {parent}/{category} ---")
            
            # Select random samples to avoid showing the same sequential images
            sample_files = random.sample(img_files, min(num_samples, len(img_files)))
            
            fig, axes = plt.subplots(1, len(sample_files), figsize=(15, 5))
            if len(sample_files) == 1:
                axes = [axes]
            
            for i, f in enumerate(sample_files):
                img_path = os.path.join(root, f)
                img = Image.open(img_path)
                axes[i].imshow(img)
                axes[i].set_title(f"{img.size[0]}x{img.size[1]}")
                axes[i].axis('off')
            plt.show()

show_samples(dataset_path)

## 4. Down-sampling for Computational Constraints
The dataset contains 180,000+ images. We can create a smaller, balanced subset for local training and experimentation to adapt to computational power.

In [ ]:
def create_downsampled_dataset(src_path, dest_path, samples_per_class=2000):
    """
    Creates a smaller, balanced dataset. It mirrors the directory structure of the source.
    """
    if not os.path.exists(dest_path):
        os.makedirs(dest_path)
        
    for root, dirs, files in os.walk(src_path):
        img_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if img_files:
            rel_path = os.path.relpath(root, src_path)
            cat_dest_path = os.path.join(dest_path, rel_path)
            
            if not os.path.exists(cat_dest_path):
                os.makedirs(cat_dest_path)
            
            sampled_files = random.sample(img_files, min(samples_per_class, len(img_files)))
            
            print(f"Copying {len(sampled_files)} images to {rel_path}...")
            for f in sampled_files:
                src_f = os.path.join(root, f)
                dest_f = os.path.join(cat_dest_path, f)
                shutil.copy2(src_f, dest_f)
    
    print(f"\nDownsampled dataset created at {os.path.abspath(dest_path)}")

local_dataset_path = "./data/downsampled_art_dataset"
# Uncomment the line below to perform down-sampling
create_downsampled_dataset(dataset_path, local_dataset_path, samples_per_class=100)